<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/JP_Morgan_Financial_Intelligence_Dashboard_using_Python_%26_Bokeh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install bokeh yfinance pandas numpy scipy

In [4]:
import pandas as pd
import numpy as np
import yfinance as yf
import os
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from math import pi

from bokeh.io import output_notebook, show, curdoc
from bokeh.plotting import figure
from bokeh.layouts import row, column
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    Div,
    Select,
    DateRangeSlider
)

In [5]:
output_notebook()

In [6]:
BACKGROUND = "#0F172A"
CARD = '#1E293B'
TEXT = '#F8FAFC'
GRID = '#334155'

WIDTH = 650
HEIGHT = 400

In [7]:
companies = {
    "JP Morgan":'JPM',
    "Goldman Sachs":'GS',
    "Morgan Stanley":'MS',
    "Bank of America":'BAC',
    'Citigroup':'C'
}

In [8]:
ticker = companies["JP Morgan"]

In [9]:
df = yf.download(
    ticker,
    period='2y',
    interval = '1d'
)

[*********************100%***********************]  1 of 1 completed


In [10]:
df.head()

Price,Close,High,Low,Open,Volume
Ticker,JPM,JPM,JPM,JPM,JPM
Date,,,,,
2024-05-09,189.341125,189.427404,187.040277,187.107378,7977300
2024-05-10,190.558655,191.105100,190.079310,190.338146,7529800
2024-05-13,190.520294,191.594036,189.858796,190.587409,7049200
2024-05-14,193.185440,193.252555,189.973840,190.779136,8596200
2024-05-15,193.760666,194.316707,191.517337,193.664791,8370000


In [11]:
df.reset_index(inplace=True)

In [12]:
df.columns = df.columns.get_level_values(0)

In [13]:
df['Returns'] = (
    df['Close'].pct_change()
)

In [15]:
df['MA20'] = (
    df['Close'].rolling(20).mean()
)

df['MA50'] = (
    df['Close'].rolling(50).mean()
)

df['MA100'] = (
    df['Close'].rolling(100).mean()
)

In [17]:
df['Volatility'] = (
    df['Returns'].rolling(20).std()
)

In [18]:
delta = df['Close'].diff()

gain = delta.where(
    delta > 0,
    0
)

loss = -delta.where(
    delta < 0,
    0
)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df['RSI'] = (
    100 - (100 / (1 + rs))
)

In [19]:
investment = 10000

shares = (
    investment
    /
    df['Close'].iloc[0]
)

df['Portfolio'] = (
    shares *
    df['Close']
)

In [21]:
df.dropna(inplace=True)

In [22]:
source = ColumnDataSource()

In [23]:
latest_price = round(
    df['Close'].iloc[-1],
    2
)

daily_return = round(
    df['Returns'].iloc[-1] * 100,
    2
)

volatility = round(
    df['Volatility'].mean() * 100,
    2
)

portfolio_value = round(
    df['Portfolio'].iloc[-1],
    2
)

profit = round(
    portfolio_value - investment,
    2
)

In [25]:
risk_free_rate = 0.02

sharpe_ratio = round(

    (
        df['Returns'].mean()*252
        - risk_free_rate
    )

    /

    (
        df['Returns'].std()
        * np.sqrt(252)
    ),

    2
)

In [26]:
def create_kpi(title, value, color):

    return Div(text=f"""

    <div style="

    background:{color};

    padding:20px;

    border-radius:18px;

    text-align:center;

    width:220px;

    height:120px;

    box-shadow:0px 4px 15px rgba(0,0,0,0.5);

    ">

    <h2 style="
    color:white;
    font-family:Arial;
    ">

    {title}

    </h2>

    <h1 style="
    color:white;
    ">

    {value}

    </h1>

    </div>

    """)

In [27]:
kpi1 = create_kpi(
    "Live Price",
    f"${latest_price}",
    "#2563EB"
)

kpi2 = create_kpi(
    "Daily Return",
    f"{daily_return}%",
    "#059669"
)

kpi3 = create_kpi(
    "Volatility",
    f"{volatility}%",
    "#DC2626"
)

kpi4 = create_kpi(
    "Portfolio Value",
    f"${portfolio_value}",
    "#7C3AED"
)

kpi5 = create_kpi(
    "Profit/Loss",
    f"${profit}",
    "#EA580C"
)

kpi6 = create_kpi(
    "Sharpe Ratio",
    sharpe_ratio,
    "#0891B2"
)

In [69]:
company_selector = Select(

    title="Select Bank",

    value="JP Morgan",

    options=list(companies.keys())

)

In [29]:
date_slider = DateRangeSlider(

    title="Select Date Range",

    start=df['Date'].min(),

    end=df['Date'].max(),

    value=(
        df['Date'].min(),
        df['Date'].max()
    ),

    width=500

)

In [30]:
inc = df['Close'] > df['Open']

dec = df['Open'] > df['Close']

w = 12 * 60 * 60 * 1000

In [31]:
p1 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="JP Morgan Candlestick Analysis",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [33]:
p1.segment(

    df['Date'],
    df['High'],

    df['Date'],
    df['Low'],

    color="white"

)

GlyphRenderer(id='p1073', ...)

In [34]:
p1.vbar(

    x=df.loc[inc,'Date'],

    width=w,

    top=df.loc[inc,'Close'],

    bottom=df.loc[inc,'Open'],

    fill_color="#10B981",

    line_color="#10B981"

)

GlyphRenderer(id='p1084', ...)

In [35]:
p1.vbar(

    x=df.loc[dec,'Date'],

    width=w,

    top=df.loc[dec,'Open'],

    bottom=df.loc[dec,'Close'],

    fill_color="#EF4444",

    line_color="#EF4444"

)

GlyphRenderer(id='p1095', ...)

In [36]:
p1.vbar(

    x=df.loc[dec,'Date'],

    width=w,

    top=df.loc[dec,'Open'],

    bottom=df.loc[dec,'Close'],

    fill_color="#EF4444",

    line_color="#EF4444"

)

GlyphRenderer(id='p1106', ...)

In [38]:
p2 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Moving Average Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [39]:
p2.line(

    df['Date'],

    df['Close'],

    line_width=2,

    color="white",

    legend_label="Close Price"

)

GlyphRenderer(id='p1166', ...)

In [41]:
p2.line(

    df['Date'],

    df['MA20'],

    line_width=3,

    color="#3B82F6",

    legend_label="MA20"

)

GlyphRenderer(id='p1179', ...)

In [42]:
p2.line(

    df['Date'],

    df['MA50'],

    line_width=3,

    color="#F59E0B",

    legend_label="MA50"
)

GlyphRenderer(id='p1191', ...)

In [43]:
p2.line(

    df['Date'],

    df['MA100'],

    line_width=3,

    color="#8B5CF6",

    legend_label="MA100"

)

GlyphRenderer(id='p1203', ...)

In [44]:
p3 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Trading Volume Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [45]:
p3.vbar(

    x=df['Date'],

    top=df['Volume'],

    width=w,

    color="#06B6D4"

)

GlyphRenderer(id='p1264', ...)

In [46]:
p4 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Volatility Risk Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [47]:
p4.line(

    df['Date'],

    df['Volatility'],

    line_width=3,

    color="#EF4444"

)

GlyphRenderer(id='p1324', ...)

In [48]:
p5 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="RSI Momentum Analysis",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [49]:
p5.line(

    df['Date'],

    df['RSI'],

    line_width=3,

    color="#F97316"

)

GlyphRenderer(id='p1384', ...)

In [50]:
p5.line(

    df['Date'],

    [70]*len(df),

    color="red",

    line_dash="dashed"

)

p5.line(

    df['Date'],

    [30]*len(df),

    color="green",

    line_dash="dashed"

)

GlyphRenderer(id='p1404', ...)

In [51]:
p6 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Portfolio Growth Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [52]:
p6.line(

    df['Date'],

    df['Portfolio'],

    line_width=4,

    color="#22C55E"

)

GlyphRenderer(id='p1464', ...)

In [53]:
hist, edges = np.histogram(

    df['Returns'].dropna(),

    bins=40

)

In [54]:
p7 = figure(

    width=WIDTH,

    height=HEIGHT,

    title="Return Distribution Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [55]:
p7.quad(

    top=hist,

    bottom=0,

    left=edges[:-1],

    right=edges[1:],

    fill_color="#8B5CF6",

    line_color="white"

)

GlyphRenderer(id='p1510', ...)

In [56]:
hover = HoverTool(

    tooltips=[

        ("Date", "@Date{%F}"),
        ("Open", "@Open"),
        ("Close", "@Close"),
        ("High", "@High"),
        ("Low", "@Low"),
        ("Volume", "@Volume")

    ],

    formatters={

        '@Date':'datetime'

    }

)

In [57]:
p1.add_tools(hover)

In [58]:
plots = [

    p1,
    p2,
    p3,
    p4,
    p5,
    p6,
    p7

]

for p in plots:

    p.title.text_color = TEXT

    p.xaxis.major_label_text_color = TEXT

    p.yaxis.major_label_text_color = TEXT

    p.xaxis.axis_label_text_color = TEXT

    p.yaxis.axis_label_text_color = TEXT

    p.grid.grid_line_color = GRID

In [59]:
title = Div(text="""

<div style="

background:linear-gradient(
90deg,
#111827,
#2563EB
);

padding:25px;

border-radius:18px;

box-shadow:0px 4px 15px rgba(0,0,0,0.5);

">

<h1 style="

color:white;

text-align:center;

font-size:40px;

font-family:Arial;

">

JP MORGAN FINANCIAL INTELLIGENCE DASHBOARD

</h1>

<p style="

color:white;

text-align:center;

font-size:18px;

">

Real-Time Banking Analytics |
Risk Analytics |
Portfolio Monitoring

</p>

</div>

""")

In [60]:
subtitle = Div(text="""

<h2 style="
color:white;
padding:10px;
">

Interactive Banking Analytics Platform

</h2>

""")

In [61]:
market_status = Div(text="""

<div style="

background:#059669;

padding:20px;

border-radius:15px;

width:250px;

text-align:center;

">

<h2 style="color:white;">

Market Status

</h2>

<h1 style="color:white;">

LIVE

</h1>

</div>

""")

In [62]:
risk_card = Div(text=f"""

<div style="

background:#DC2626;

padding:20px;

border-radius:15px;

width:250px;

text-align:center;

box-shadow:0px 4px 12px rgba(0,0,0,0.4);

">

<h2 style="color:white;">

Risk Level

</h2>

<h1 style="color:white;">

{round(volatility,2)}%

</h1>

</div>

""")

In [67]:
return_card = Div(text=f"""

<div style="

background:#7C3AED;

padding:20px;

border-radius:15px;

width:250px;

text-align:center;

box-shadow:0px 4px 12px rgba(0,0,0,0.4);

">

<h2 style="color:white;">

Expected Return

</h2>

<h1 style="color:white;">

{daily_return}%

</h1>

</div>

""")

In [63]:
divider = Div(text="""

<hr style="
border:2px solid #334155;
">

""")

In [64]:
latest = yf.download(

    ticker,

    period="1d",

    interval="1m"

    )

[*********************100%***********************]  1 of 1 completed


In [65]:
latest_price = latest['Close'].iloc[-1]

In [70]:
dashboard = column(

    title,

    subtitle,

    divider,

    row(

        kpi1,
        kpi2,
        kpi3

    ),

    row(

        kpi4,
        kpi5,
        kpi6

    ),

    row(

        market_status,
        risk_card,
        return_card

    ),

    divider,

    row(

        company_selector,
        date_slider

    ),

    row(

        p1,
        p2

    ),

    row(

        p3,
        p4

    ),

    row(

        p5,
        p6

    ),

    p7

)

In [71]:
show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


In [72]:
from bokeh.io import output_file, save

output_file("dashboard.html")

save(dashboard)

show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


In [73]:
from bokeh.io import output_file, save
output_file("netflix_dashboard.html")
show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


In [74]:
from bokeh.io import output_file, save
output_file("JP Morgan_dashboard.html")
show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


In [75]:
from bokeh.io import output_file, save
output_file("JP Morgan_dashboard.html")
show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1535', ...)


In [76]:
from google.colab import files

files.download("JP Morgan_dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>